# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/leoxie/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-20 22:26:58.678009: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [5]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df[feature_names].values
y = df["Class"].values
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))


X shape: (178, 13)
y shape: (178,)
Unique labels: [0 1 2]


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("X_train:", X_train.shape, "X_test:", X_test.shape)


X_train: (124, 13) X_test: (54, 13)


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)


In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)
print("y_train_cat shape:", y_train_cat.shape)


y_train_cat shape: (124, 3)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

tf.random.set_seed(42)
np.random.seed(42)

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax'),
])
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20, batch_size=8, validation_split=0.2, verbose=2,
)

train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
print(f"\nTraining accuracy: {train_acc:.4f}")


Epoch 1/20
13/13 - 1s - loss: 1.0315 - accuracy: 0.5657 - val_loss: 0.8144 - val_accuracy: 0.7600 - 649ms/epoch - 50ms/step
Epoch 2/20
13/13 - 0s - loss: 0.7176 - accuracy: 0.8586 - val_loss: 0.5589 - val_accuracy: 1.0000 - 37ms/epoch - 3ms/step
Epoch 3/20
13/13 - 0s - loss: 0.4950 - accuracy: 0.9798 - val_loss: 0.3915 - val_accuracy: 1.0000 - 34ms/epoch - 3ms/step
Epoch 4/20
13/13 - 0s - loss: 0.3380 - accuracy: 0.9899 - val_loss: 0.2774 - val_accuracy: 0.9600 - 38ms/epoch - 3ms/step
Epoch 5/20
13/13 - 0s - loss: 0.2364 - accuracy: 0.9899 - val_loss: 0.2017 - val_accuracy: 0.9600 - 31ms/epoch - 2ms/step
Epoch 6/20
13/13 - 0s - loss: 0.1684 - accuracy: 0.9899 - val_loss: 0.1548 - val_accuracy: 0.9600 - 31ms/epoch - 2ms/step
Epoch 7/20
13/13 - 0s - loss: 0.1264 - accuracy: 0.9899 - val_loss: 0.1254 - val_accuracy: 0.9600 - 35ms/epoch - 3ms/step
Epoch 8/20
13/13 - 0s - loss: 0.0962 - accuracy: 0.9899 - val_loss: 0.1086 - val_accuracy: 0.9600 - 32ms/epoch - 2ms/step
Epoch 9/20
13/13 - 0s 

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nClassification report:")
print(classification_report(y_true, y_pred))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))


Test accuracy: 1.0000

Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

import os

def file_size_kb(path):
    return os.path.getsize(path) / 1024.0

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_base = converter.convert()
with open("model_base.tflite", "wb") as f:
    f.write(tflite_base)

print(f"Baseline (float32) TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpfh5oczkf/assets


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpfh5oczkf/assets


Baseline (float32) TFLite model size: 14.07 KB


2026-05-20 22:27:05.254512: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:27:05.254528: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:27:05.254835: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpfh5oczkf
2026-05-20 22:27:05.255759: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:27:05.255770: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpfh5oczkf
2026-05-20 22:27:05.257571: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 22:27:05.258296: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:27:05.290127: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.

        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    in_scale, in_zp = input_details['quantization']
    out_scale, out_zp = output_details['quantization']
    in_dtype = input_details['dtype']
    out_dtype = output_details['dtype']

    y_pred = []
    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)
        if in_scale != 0:
            sample = (sample / in_scale + in_zp).astype(in_dtype)
        else:
            sample = sample.astype(in_dtype)
        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        if out_scale != 0:
            output = (output.astype(np.float32) - out_zp) * out_scale
        y_pred.append(np.argmax(output, axis=1)[0])

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    acc = np.mean(y_pred == y_true)
    print(f"{quant_type.upper()} accuracy: {acc:.4f}")
    print("Classification report:")
    print(classification_report(y_true, y_pred))
    print("Confusion matrix:")
    print(confusion_matrix(y_true, y_pred))


In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpwdxa30_m/assets


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpwdxa30_m/assets
2026-05-20 22:27:05.976863: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:27:05.976881: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:27:05.977073: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpwdxa30_m
2026-05-20 22:27:05.978002: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:27:05.978013: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpwdxa30_m
2026-05-20 22:27:05.980507: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:27:06.015148: I tensorflow/cc/saved_model/loader.cc:217] Running initialization


DYNAMIC TFLite model size: 8.17 KB
DYNAMIC accuracy: 1.0000
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmp6vztgjk2/assets


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmp6vztgjk2/assets
/Users/leoxie/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 22:27:06.548465: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:27:06.548489: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:27:06.548723: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmp6vztgjk2
2026-05-20 22:27:06.549614: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:27:06.549625: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ds/vqtbdcsn7xlbn0


INT8 TFLite model size: 5.74 KB
INT8 accuracy: 1.0000
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]
INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpeux3yi6r/assets


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpeux3yi6r/assets



FLOAT16 TFLite model size: 8.95 KB
FLOAT16 accuracy: 1.0000
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
[[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


2026-05-20 22:27:07.156985: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:27:07.157021: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:27:07.157246: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpeux3yi6r
2026-05-20 22:27:07.158493: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:27:07.158503: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpeux3yi6r
2026-05-20 22:27:07.161398: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:27:07.193689: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpeux3yi6r
2026-05-

## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

batch_size = 8
epochs_prune = 10
num_train_samples = int(X_train_scaled.shape[0] * 0.8)  # accounting for validation_split=0.2
end_step = int(np.ceil(num_train_samples / batch_size) * epochs_prune)

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step,
)
print("end_step:", end_step)


end_step: 130


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

tf.random.set_seed(42)
np.random.seed(42)

pruning_params = {'pruning_schedule': pruning_schedule}

pruned_model = Sequential([
    prune_low_magnitude(Dense(64, activation='relu', input_shape=(num_features,)), **pruning_params),
    prune_low_magnitude(Dense(32, activation='relu'), **pruning_params),
    prune_low_magnitude(Dense(num_classes, activation='softmax'), **pruning_params),
])
pruned_model.summary()


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

pruned_model.fit(
    X_train_scaled, y_train_cat,
    epochs=epochs_prune, batch_size=batch_size, validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=2,
)


Epoch 1/10
13/13 - 2s - loss: 1.0858 - accuracy: 0.4343 - val_loss: 0.8419 - val_accuracy: 0.8800 - 2s/epoch - 125ms/step
Epoch 2/10
13/13 - 0s - loss: 0.7481 - accuracy: 0.8485 - val_loss: 0.5780 - val_accuracy: 0.9600 - 37ms/epoch - 3ms/step
Epoch 3/10
13/13 - 0s - loss: 0.5226 - accuracy: 0.9596 - val_loss: 0.4073 - val_accuracy: 0.9600 - 37ms/epoch - 3ms/step
Epoch 4/10
13/13 - 0s - loss: 0.3720 - accuracy: 0.9596 - val_loss: 0.2850 - val_accuracy: 0.9600 - 34ms/epoch - 3ms/step
Epoch 5/10
13/13 - 0s - loss: 0.2613 - accuracy: 0.9697 - val_loss: 0.2029 - val_accuracy: 0.9600 - 36ms/epoch - 3ms/step
Epoch 6/10
13/13 - 0s - loss: 0.1832 - accuracy: 0.9899 - val_loss: 0.1527 - val_accuracy: 0.9600 - 36ms/epoch - 3ms/step
Epoch 7/10
13/13 - 0s - loss: 0.1351 - accuracy: 0.9899 - val_loss: 0.1204 - val_accuracy: 0.9600 - 35ms/epoch - 3ms/step
Epoch 8/10
13/13 - 0s - loss: 0.2068 - accuracy: 0.9697 - val_loss: 0.4910 - val_accuracy: 0.8400 - 36ms/epoch - 3ms/step
Epoch 9/10
13/13 - 0s - 

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_pruned = converter.convert()
with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpldkfvtkh/assets


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpldkfvtkh/assets


Pruned TFLite model size: 8.24 KB


2026-05-20 22:27:10.177032: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:27:10.177047: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:27:10.177227: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpldkfvtkh
2026-05-20 22:27:10.177960: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:27:10.177971: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpldkfvtkh
2026-05-20 22:27:10.179537: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:27:10.195013: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpldkfvtkh
2026-05-

In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred_pruned = np.argmax(stripped_model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

acc_pruned = np.mean(y_pred_pruned == y_true)
print(f"Pruned model test accuracy: {acc_pruned:.4f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred_pruned))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred_pruned))


Pruned model test accuracy: 0.9444

Classification report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.91      0.95      0.93        21
           2       1.00      0.87      0.93        15

    accuracy                           0.94        54
   macro avg       0.95      0.94      0.94        54
weighted avg       0.95      0.94      0.94        54

Confusion matrix:
[[18  0  0]
 [ 1 20  0]
 [ 0  2 13]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

tf.random.set_seed(42)
np.random.seed(42)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax'),
])
student_model.summary()


Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled, verbose=0)
print("Teacher soft labels shape:", teacher_preds_soft.shape)


Teacher soft labels shape: (124, 3)


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)
print("Combined label shape:", y_train_combined.shape)

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :num_classes]
    y_true_soft = y_true_combined[:, num_classes:]
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)
    return alpha * hard_loss + (1.0 - alpha) * soft_loss


Combined label shape: (124, 6)


In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(optimizer='adam', loss=distillation_loss, metrics=['accuracy'])

student_model.fit(
    X_train_scaled, y_train_combined,
    epochs=10, batch_size=8, validation_split=0.2, verbose=2,
)


Epoch 1/10
13/13 - 1s - loss: 1.0319 - accuracy: 0.4747 - val_loss: 0.9671 - val_accuracy: 0.6400 - 925ms/epoch - 71ms/step
Epoch 2/10
13/13 - 0s - loss: 0.9196 - accuracy: 0.7172 - val_loss: 0.8753 - val_accuracy: 0.9200 - 34ms/epoch - 3ms/step
Epoch 3/10
13/13 - 0s - loss: 0.8280 - accuracy: 0.8081 - val_loss: 0.7929 - val_accuracy: 0.9200 - 33ms/epoch - 3ms/step
Epoch 4/10
13/13 - 0s - loss: 0.7452 - accuracy: 0.8586 - val_loss: 0.7156 - val_accuracy: 0.9200 - 32ms/epoch - 2ms/step
Epoch 5/10
13/13 - 0s - loss: 0.6616 - accuracy: 0.9091 - val_loss: 0.6395 - val_accuracy: 0.9600 - 33ms/epoch - 3ms/step
Epoch 6/10
13/13 - 0s - loss: 0.5702 - accuracy: 0.9495 - val_loss: 0.5569 - val_accuracy: 0.9600 - 34ms/epoch - 3ms/step
Epoch 7/10
13/13 - 0s - loss: 0.4829 - accuracy: 0.9596 - val_loss: 0.4778 - val_accuracy: 0.9600 - 33ms/epoch - 3ms/step
Epoch 8/10
13/13 - 0s - loss: 0.3993 - accuracy: 0.9798 - val_loss: 0.4179 - val_accuracy: 0.9600 - 33ms/epoch - 3ms/step
Epoch 9/10
13/13 - 0s 

In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()
with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd)

print(f"Knowledge-distilled student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpnb1cf3m7/assets


INFO:tensorflow:Assets written to: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpnb1cf3m7/assets


Knowledge-distilled student TFLite model size: 6.10 KB


2026-05-20 22:27:12.336364: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 22:27:12.336379: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 22:27:12.336590: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpnb1cf3m7
2026-05-20 22:27:12.337368: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 22:27:12.337377: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpnb1cf3m7
2026-05-20 22:27:12.339695: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 22:27:12.368554: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/ds/vqtbdcsn7xlbn06mfwj3gzs40000gp/T/tmpnb1cf3m7
2026-05-

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_student = np.argmax(student_model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

acc_student = np.mean(y_pred_student == y_true)
print(f"Student test accuracy: {acc_student:.4f}")
print("\nClassification report:")
print(classification_report(y_true, y_pred_student))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred_student))


Student test accuracy: 0.8704

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.89      0.91        18
           1       0.89      0.76      0.82        21
           2       0.79      1.00      0.88        15

    accuracy                           0.87        54
   macro avg       0.87      0.88      0.87        54
weighted avg       0.88      0.87      0.87        54

Confusion matrix:
[[16  2  0]
 [ 1 16  4]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [27]:
# <-- (if needed) Enter your code here <--#

# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
